In [2]:
import sys
sys.path.append(r"c:\main\GitHub\handwrittenTextRecognitionSystem")

In [3]:
from typing import Any, Tuple, Dict, List, Optional

from tqdm import tqdm

import torch
import torch.nn as nn
import numpy as np

import datasets
from datasets import load_dataset
from transformers import AutoTokenizer

from models.transformers import TransformerEncoder

from torch.utils.data import Dataset, DataLoader

In [4]:
torch.cuda.is_available()

True

### Enocder training on text classification task

In [5]:
emotion = load_dataset('emotion', split='train')
emotion

Dataset({
    features: ['text', 'label'],
    num_rows: 16000
})

In [157]:
small_emotion = datasets.Dataset.from_dict(emotion[:50])

In [158]:
print(f"Dataset type: {type(emotion)}")
print(f"Dataset length: {len(emotion)}")

Dataset type: <class 'datasets.arrow_dataset.Dataset'>
Dataset length: 16000


In [159]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [160]:
text_sample = emotion["text"][15]
text_sample

'i do not feel reassured anxiety is on each side'

In [161]:
input_ids = tokenizer(text_sample)["input_ids"]
input_ids

[101, 1045, 2079, 2025, 2514, 26350, 10089, 2003, 2006, 2169, 2217, 102]

In [162]:
tokenizer.decode(input_ids)

'[CLS] i do not feel reassured anxiety is on each side [SEP]'

In [163]:
class EmotionDataset(Dataset):
    def __init__(self, data: datasets.arrow_dataset.Dataset):
        self.data = data
        self.tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
        self.max_length = None
    
    def get_max_length(self) -> None:
        """
        Gets the maximum length of sequence in Dataset 
        """
        max_length = 0
        for row in self.data:
            text = row["text"]
            text_ids = self.tokenizer(text)["input_ids"]
            max_length = max(max_length, len(text_ids))
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[torch.tensor, torch.tensor]:
        if self.max_length is None:
            self.get_max_length()

        row = self.data[idx]
        text = row["text"]
        label = row["label"]
        text_ids = self.tokenizer(text, padding="max_length", max_length=self.max_length)["input_ids"]
        return torch.tensor(text_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [195]:
emd = EmotionDataset(data=emotion)

In [196]:
print(f"Object example: ")
emd[3]

Object example: 


(tensor([  101,  1045,  2572,  2412,  3110, 16839,  9080, 12863,  2055,  1996,
         13788,  1045,  2097,  2113,  2008,  2009,  2003,  2145,  2006,  1996,
          3200,   102,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]),
 tensor(2))

In [197]:
print(f"Max sequence length: {emd.max_length}")

Max sequence length: 87


In [198]:
value = tuple(lst.tolist() for lst in np.unique(np.array(small_emotion['label']), return_counts=True))
print('Target distribution:')
value

Target distribution:


([0, 1, 2, 3, 4, 5], [13, 19, 3, 9, 4, 2])

In [199]:
class EncoderTestModel(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, d_model:int,
                 d_feedforward: int, h: int, num_layers: int, dropout: float = 0.35):
        super().__init__()
        self.trans_encoder = TransformerEncoder(vocab_size=vocab_size,
                                                max_seq_len = max_seq_len,
                                                d_model = d_model,
                                                d_feedforward = d_feedforward,
                                                h = h,
                                                num_layers=num_layers,
                                                dropout=dropout)
        self.lin = nn.Linear(d_model, vocab_size)
    
    def forward(self, x: torch.tensor):
        """
        x: torch.tensor
            Tensor with vocabulary indecies of shape BxS 
        """
        x = self.trans_encoder(x)
        x = self.lin(x)
        return x

In [200]:
vocab_size = len(emd.tokenizer.get_vocab())
max_seq_len = emd.max_length
d_model = 256
d_feedforward = 512
h = 4
num_layers = 1
dropout = 0.25

In [201]:
encoder_test_model = EncoderTestModel(vocab_size=vocab_size,
                                      max_seq_len=max_seq_len,
                                      d_model=d_model,
                                      d_feedforward=d_feedforward,
                                      h=h,
                                      num_layers=num_layers,
                                      dropout=dropout)
encoder_test_model

EncoderTestModel(
  (trans_encoder): TransformerEncoder(
    (token_embed): Embedding(30522, 256, padding_idx=0)
    (pos_embed): Embedding(87, 256)
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (mha): MultiHeadAttention(
          (heads_dict): ModuleDict(
            (head_0): ModuleDict(
              (q): Linear(in_features=256, out_features=256, bias=True)
              (k): Linear(in_features=256, out_features=256, bias=True)
              (v): Linear(in_features=256, out_features=256, bias=True)
            )
            (head_1): ModuleDict(
              (q): Linear(in_features=256, out_features=256, bias=True)
              (k): Linear(in_features=256, out_features=256, bias=True)
              (v): Linear(in_features=256, out_features=256, bias=True)
            )
            (head_2): ModuleDict(
              (q): Linear(in_features=256, out_features=256, bias=True)
              (k): Linear(in_features=256, out_features=256, bias=True)
            

In [204]:
test_inputs = torch.randint(0, 30_000, size=(32, 87))
test_inputs.shape

torch.Size([32, 87])

In [205]:
print('Output shape: ')
encoder_test_model(test_inputs).shape

Output shape: 


torch.Size([32, 87, 30522])

In [206]:
loader = DataLoader(dataset=emd, batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=encoder_test_model.parameters(), lr=3e-5)

In [207]:
def train_epoch(
    model: torch.nn.Module,
    dataloader: DataLoader,
    criterion: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> List[float]:
    """Execute one training epoch."""
    model.train()
    loss_list = []
    
    for inputs, targets in tqdm(dataloader, desc="Going through the dataset"):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)[:, 0, :]
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        loss_value = round(loss.detach().item(), 5)
        loss_list.append(loss_value)
    return loss_list

In [208]:
def train_model(
    model: torch.nn.Module,
    train_loader: DataLoader,
    criterion: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: torch.device,
) -> Dict[str, list]:
    """Main training loop."""
    avg_epoch_loss = []
    
    for epoch in range(num_epochs):
        # Training
        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        mean_loss = np.array(train_loss).mean()
        avg_epoch_loss.append(mean_loss)
        print(mean_loss)
    
    return avg_epoch_loss

In [209]:
encoder_test_model.to(device='cuda')

EncoderTestModel(
  (trans_encoder): TransformerEncoder(
    (token_embed): Embedding(30522, 256, padding_idx=0)
    (pos_embed): Embedding(87, 256)
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (mha): MultiHeadAttention(
          (heads_dict): ModuleDict(
            (head_0): ModuleDict(
              (q): Linear(in_features=256, out_features=256, bias=True)
              (k): Linear(in_features=256, out_features=256, bias=True)
              (v): Linear(in_features=256, out_features=256, bias=True)
            )
            (head_1): ModuleDict(
              (q): Linear(in_features=256, out_features=256, bias=True)
              (k): Linear(in_features=256, out_features=256, bias=True)
              (v): Linear(in_features=256, out_features=256, bias=True)
            )
            (head_2): ModuleDict(
              (q): Linear(in_features=256, out_features=256, bias=True)
              (k): Linear(in_features=256, out_features=256, bias=True)
            

In [210]:
result = train_model(
    model = encoder_test_model,
    train_loader = loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=10,
    device='cuda'
)

Going through the dataset: 100%|██████████| 500/500 [00:51<00:00,  9.62it/s]


4.59889832


Going through the dataset: 100%|██████████| 500/500 [00:52<00:00,  9.53it/s]


1.80892002


Going through the dataset: 100%|██████████| 500/500 [00:52<00:00,  9.56it/s]


...

1.5872538600000001


Going through the dataset: 100%|██████████| 500/500 [00:52<00:00,  9.52it/s]


1.58267958


Going through the dataset: 100%|██████████| 500/500 [00:51<00:00,  9.65it/s]

1.57590986
